# 실시간 side plate 6D pose: Photoneo 촬영 → robot base 기준 pick 10개

`results_photoneo_summary.ipynb`에서 검증한 **가장 성능이 좋은 설정**만 뽑아, Photoneo로 한 장을 찍고 바로 side plate pose를 냅니다.

**Photoneo 촬영 (1236×1032, 렌즈 왜곡 보정) → Mask R-CNN detector → GDRN 방식 RGB-D pose 네트워크 → 깊이 맞춤 → ICP → 중복 제거 → 깊이 검증 → confidence 판정 → 3가지 조건으로 10개 선정 → robot base 좌표**

| 단계 | 설정 | 근거 (summary 노트북) |
|---|---|---|
| 촬영 | PrimaryCamera 1236×1032 (학습과 같은 카메라 공간), 왜곡 보정 | §7: 왜곡 보정 없는 촬영은 이미지 가장자리 성능이 떨어짐 |
| detector | `runs/detector_ft/best.pt` (시뮬레이션 + 실제 fine-tuning) | §6: 평가용 23장에서 찾은 부품 94% → 100%, 오검출 0 |
| pose | `runs/pose/best.pt` (시뮬레이션 학습) | §8: fine-tuning한 pose는 최종 결과를 바꾸지 않음 |
| 정제 | 깊이 맞춤 + ICP + 중복 제거 + 깊이 검증 (`configs/default.yaml`의 `refine`) | §3~§4: 출력에서 집으면 안 되는 오검출 0개 |

**10개 선정 (summary §10과 같은 규칙):** 아래 3가지 조건을 모두 만족하는 부품을 confidence score 순으로 최대 10개 고르고, 고른 부품끼리는 이미지에서 겹치지 않게 합니다.
1. **맨 위:** confident이고 대체로 덮이지 않음 (occluded ≤ 0.5)
2. **confidence 높음:** score = support × fitness × min(1, 프레임 맞춤 오차 중앙값 / 이 부품의 맞춤 오차). 이 순서가 pick 우선순위입니다.
3. **집을 수 있음:** 접근 기울기 ≤ `MAX_TILT`(60°, 마지막 셀), 가운데 구멍과 그리퍼 몸체 주변이 비어 있음. 기울어진 부품도 툴 z축을 부품 축을 따라 아래로 향하게 해서 집습니다

**출력 = robot TCP 목표 pose (robot base, m):** TCP(손끝)는 위쪽 끝면의 가운데 구멍 입구에서 8 mm 들어간 점으로 갑니다. 툴 z축은 부품 축을 따라 **아래 방향**입니다(부품 z가 위면 −z, 아래면 +z, 기울어져도 같은 규칙). 위치, rpy(extrinsic/intrinsic), 쿼터니언, 4×4 행렬과 부품 자체의 자세를 같이 냅니다.

**10개를 한 번에 집는 가정:** 10개는 **한 번 촬영한 결과**입니다. 로봇은 1번부터 10번까지 순서대로 집고, 그동안 나머지 부품이 **움직이지 않는다고 가정**합니다.
고른 부품끼리 겹치지 않고 각 부품의 구멍과 그리퍼 몸체 주변이 비어 있어야 해서, 앞 순서를 집을 때 뒤 순서가 밀릴 가능성을 줄입니다. 다시 찍으려면 마지막 셀을 다시 실행하세요.

**사용법**
1. hand-eye 캘리브레이션 결과(카메라 → robot base, 4×4, **m**)는 **마지막 셀의 `T_CAM_BASE`**에 있습니다. 캘리브레이션을 다시 하면 그 값만 바꾸세요. 접근 기울기는 base +Z를 위쪽으로 보고 잽니다.
2. 처음 한 번 위에서부터 모두 실행합니다(모델 로딩 약 10초). 그다음은 **마지막 셀만 다시 실행**하면 새로 찍고 pose를 냅니다.
3. 결과는 `runs/realtime/<시각>/`에 저장됩니다(`picks.json`, `picks.csv`, `overlay.png`, `rgb.png`, `depth.npy`, `detections.json`).

카메라 없이 확인하려면 `OFFLINE = ("data/pseudo_gt_70_bop", 59)`처럼 저장된 BOP 프레임을 지정하세요.

In [ ]:
import os, subprocess, sys, json, time
from datetime import datetime
from pathlib import Path
if Path.cwd().name == "notebooks":                      # run from the project root
    os.chdir(Path.cwd().parent)
sys.path.insert(0, str(Path.cwd()))
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")
os.environ.setdefault("HF_HUB_OFFLINE", "1")

import numpy as np, pandas as pd, torch
from IPython.display import display
from PIL import Image, ImageDraw, ImageFont
from scipy.ndimage import binary_closing, binary_fill_holes
from scipy.spatial.transform import Rotation

from data_conversion.bin_dataset import BinPoseDataset
from bin6d.config import Cfg, load_config
from bin6d.detector import build_detector, set_native_scale
from bin6d.pose_data import ObjectModel, frame_planes, input_channels
from bin6d.pose_model import GDRNet
from bin6d.refine import (dedup_detections, depth_anchor, flag_confident, icp_refine, mask_cover, observed_points,
                          reject_unverified)
from bin6d.report import load_frame, frame_index, surface_check
from bin6d.viz import render_2d
from data_conversion.camera_convert_data import undistort, undistort_lookup
from infer import predict_poses
from torchvision.ops import masks_to_boxes

# camera
HARVESTERS_PY = Path.home() / "Projects/photoneo-python-examples/GigE-V/harvesters/.venv/bin/python"
SERIAL = None                                 # Photoneo serial number; None = first device found
CAPTURE_DIR = Path("runs/realtime/capture")   # raw captures (camera_convert_data.py layout)
UNDISTORT = True                              # resample rgb/depth onto the pinhole K (same as --to-bop)
OFFLINE = None                                # (bop_root, view) = use a saved frame instead of the camera

# robot
T_BASE_CAM_FILE = Path("camera/T_base_cam.json")   # {"T_base_cam": 4x4}, camera -> robot base, mm

# models: best setting of results_photoneo_summary.ipynb (§6-§9)
CONFIG = "configs/default.yaml"
DETECTOR = "runs/detector_ft/best.pt"         # sim + real fine-tuned detector
POSE = "runs/pose/best.pt"                    # sim-trained pose (fine-tuned pose: same final result, §8)
MODEL_ROOT = "data/pseudo_gt_70_bop"          # CAD model + symmetries (same CAD as training)
DET_SCALE = 1.0                               # detector input scale; 1.0 for the 1236x1032 primary camera
ROI_PAD = 8                                   # px around each mask for depth refinement (> mask_erode_px, mask_cover dilation)

# pick selection (summary §10)
TOP_K = 10
R_FINGER = 5.0          # mm, radius of the closed fingers (hole: 11.2-13.8 mm diameter)
INSERT_DEPTH = 8.0      # mm, finger insertion depth
R_BODY = 15.0           # mm, radius of the gripper body / tip that must clear the surroundings
RING_MARGIN = 3.0       # mm, surroundings may stick up this much above the hole entrance (noise, calibration)
APPROACH_LEN = 60.0     # mm, height above the entrance checked for obstacles
HOLE_BLOCK_MAX = 0.10   # allowed blocked share of the finger disc
RING_BLOCK_MAX = 0.10   # allowed blocked share of the body ring
GRASP_MAX_OCCLUDED = 0.5
MAX_OVERLAP_SELECTED = 0.02
OUT_DIR = Path("runs/realtime")
pd.set_option("display.width", 220)

## 1. 모델과 robot base 변환 불러오기 (한 번만)

In [ ]:
def load_T_base_cam(path):
    if not Path(path).exists():
        return None
    T = np.asarray(json.load(open(path))["T_base_cam"], float)
    if T.shape != (4, 4) or not np.allclose(T[3], [0, 0, 0, 1]) or not np.allclose(T[:3, :3] @ T[:3, :3].T, np.eye(3), atol=1e-3):
        raise ValueError(f"{path}: T_base_cam must be a rigid 4x4 transform (camera -> base, mm)")
    if np.linalg.norm(T[:3, 3]) < 10:
        print(f"WARNING: |t| of T_base_cam is {np.linalg.norm(T[:3, 3]):.3f}: looks like metres, the pipeline uses mm")
    return T

T_BASE_CAM = load_T_base_cam(T_BASE_CAM_FILE)
if T_BASE_CAM is None:
    FRAME, T_OUT = "cam", np.eye(4)
    print(f"WARNING: {T_BASE_CAM_FILE} not found -> poses are in the CAMERA frame, not robot base")
else:
    FRAME, T_OUT = "base", T_BASE_CAM
    print(f"T_base_cam ({T_BASE_CAM_FILE}):\n{np.array2string(T_BASE_CAM, precision=3, suppress_small=True)}")
UP_CAM = T_OUT[2, :3] if T_BASE_CAM is not None else np.array([0.0, 0.0, -1.0])   # "up" (out of the bin) in the camera frame

cfg = load_config(CONFIG)
rc, ic = cfg.refine, cfg.infer
device = torch.device(cfg.device)
ck_pose = torch.load(POSE, map_location=device, weights_only=False)
pc = Cfg(ck_pose["cfg"]).pose                  # architecture the checkpoint was trained with
pc.pretrained = False
net = GDRNet(pc, input_channels()).to(device).eval()
net.load_state_dict(ck_pose["model"])
ck_det = torch.load(DETECTOR, map_location=device, weights_only=False)
det = build_detector(Cfg(ck_det["cfg"]).detector).to(device).eval()
det.load_state_dict(ck_det["model"])
det.roi_heads.score_thresh = ic.score_thresh
model = ObjectModel(BinPoseDataset(MODEL_ROOT), cfg.data.obj_id, pc.num_regions, pc.num_model_points)
print(f"detector {DETECTOR} | pose {POSE} | part diameter {model.diameter:.1f} mm | device {device}")
print(f"refine: depth_anchor={rc.depth_anchor}, dedup_mm={rc.dedup_mm}, reject_unverified={rc.reject_unverified}")

## 2. 촬영, 추론, 10개 선정 함수

- **촬영:** Photoneo 드라이버(harvesters)는 별도 venv에만 설치되어 있어서 그 venv의 python으로 `camera_convert_data.py --count 1`을 실행합니다.
  같은 세션의 왜곡 보정 맵(`undistort_map.npy`)으로 rgb와 depth를 핀홀 K에 맞게 다시 샘플링합니다(`--to-bop`과 같은 처리).
- **추론:** `infer.py`의 한 프레임 처리와 같습니다(정답 비교만 뺌). 깊이 맞춤, ICP, 검증은 각 마스크의 bbox 영역만 잘라서 계산합니다.
  결과는 전체 이미지로 계산한 것과 같고(Photoneo 59번, 가득 찬 bin 0번에서 확인), 가득 찬 bin에서 약 3배 빠릅니다.
- **선정:** summary §10의 `top_picks`와 같습니다. 접근 기울기만 base +Z 기준으로 잽니다(`T_base_cam`이 있을 때).

In [ ]:
_lookups = {}

def _undistort_frame(rgb, depth, cam_dir, cam):
    key = (cam["width"], cam["height"], round(cam["fx"], 3), round(cam["fy"], 3), round(cam["cx"], 3), round(cam["cy"], 3))
    if key not in _lookups:
        ab = np.load(Path(cam_dir) / cam["undistort_map"])
        if ab.shape[:2] != (cam["height"], cam["width"]):
            raise ValueError(f"undistort map {ab.shape[:2]} does not match camera.json {cam['height']}x{cam['width']}")
        _lookups[key] = undistort_lookup(ab)
    lk = _lookups[key]
    return undistort(rgb, *lk, order=1), undistort(depth, *lk, order=0)

def capture_frame():
    """One Photoneo shot -> {rgb, depth (mm), K, xyz, valid, source}."""
    if OFFLINE is not None:
        root, view = OFFLINE
        ds = BinPoseDataset(root)
        f = load_frame(ds, frame_index(ds)[(ds.frames[0].scene, view)])
        return {k: f[k] for k in ("rgb", "depth", "K", "xyz", "valid")} | {"source": f"{root} view {view}"}
    before = {p.name for p in (CAPTURE_DIR / "rgb").glob("*.png")}
    cmd = [str(HARVESTERS_PY), "data_conversion/camera_convert_data.py", "--out", str(CAPTURE_DIR), "--count", "1"] + ([SERIAL] if SERIAL else [])
    subprocess.run(cmd, check=True)
    new = sorted({p.name for p in (CAPTURE_DIR / "rgb").glob("*.png")} - before)
    if not new:
        raise RuntimeError("camera_convert_data.py wrote no new frame")
    cam = json.load(open(CAPTURE_DIR / "camera.json"))
    if (cam["width"], cam["height"]) != (1236, 1032):
        print(f"WARNING: capture is {cam['width']}x{cam['height']}, models were trained on 1236x1032 (set DET_SCALE)")
    rgb = np.asarray(Image.open(CAPTURE_DIR / "rgb" / new[-1]).convert("RGB"))
    depth = np.asarray(Image.open(CAPTURE_DIR / "depth" / new[-1]), dtype=np.float32) * cam["depth_scale"]
    if UNDISTORT:
        rgb, depth = _undistort_frame(rgb, depth, CAPTURE_DIR, cam)
    K = np.array([[cam["fx"], 0, cam["cx"]], [0, cam["fy"], cam["cy"]], [0, 0, 1]], np.float32)
    from data_conversion.bin_dataset import backproject
    return {"rgb": rgb, "depth": depth, "K": K, "xyz": backproject(depth, K), "valid": depth > 0,
            "source": str(CAPTURE_DIR / "rgb" / new[-1])}

@torch.no_grad()
def estimate_poses(f):
    """infer.py on one frame without GT: detect -> GDRN -> depth anchor -> ICP -> dedup -> verify -> confidence."""
    rgb, depth, K_np = f["rgb"], f["depth"], np.asarray(f["K"], np.float32)
    rgb_t, xyz_t, valid_t = (x.to(device) for x in frame_planes(rgb, depth, K_np))
    K = torch.from_numpy(K_np).to(device)
    h, w = depth.shape
    set_native_scale(det, round(h * DET_SCALE), round(w * DET_SCALE))
    o = det([rgb_t])[0]
    masks = o["masks"][:, 0] >= 0.5
    scores = o["scores"].cpu().numpy()
    big = masks.flatten(1).sum(1) >= ic.min_mask_px
    masks, scores = masks[big], scores[big.cpu().numpy()]
    if len(masks) == 0:
        return [], [], 0
    R, t = predict_poses(net, pc, model, rgb_t, xyz_t, valid_t, masks, K, device, ic.pose_batch)
    # depth refinement on each mask's bbox (+ ROI_PAD px) with K shifted to the crop: same result as the full image
    # (everything outside the bbox is off the mask), but no full-resolution mask copies (~4x faster on full bins)
    boxes = masks_to_boxes(masks).int().cpu().numpy()
    crops, offsets, results = [], [], []
    for n, (x0, y0, x1, y1) in enumerate(boxes):
        y0, x0 = max(y0 - ROI_PAD, 0), max(x0 - ROI_PAD, 0)
        y1, x1 = min(y1 + ROI_PAD + 1, h), min(x1 + ROI_PAD + 1, w)
        m = masks[n, y0:y1, x0:x1].cpu().numpy()
        crops.append(m)
        offsets.append((y0, x0))
        Kc = K_np.copy()
        Kc[0, 2] -= x0
        Kc[1, 2] -= y0
        t0 = (depth_anchor(R[n], t[n], depth[y0:y1, x0:x1], m, Kc, model.dense_points, model.dense_normals,
                           rc.depth_anchor_min_px) if rc.get("depth_anchor") else t[n])
        ref = icp_refine(R[n], t0, observed_points(f["xyz"][y0:y1, x0:x1], f["valid"][y0:y1, x0:x1], m, rc.mask_erode_px),
                         model.dense_points, model.dense_normals, rc)
        results.append({"score": float(scores[n]), "R_rgbd": R[n].tolist(), "t_rgbd": t[n].tolist(),
                        "t_anchored": np.asarray(t0).tolist(),
                        "R": np.asarray(ref["R"]).tolist(), "t": np.asarray(ref["t"]).tolist(),
                        "icp_refined": ref["refined"], "icp_fitness": ref["fitness"], "icp_rmse": ref["rmse"],
                        "n_depth_points": ref["n_points"], "mask_idx": n, "mask_px": int(m.sum()),
                        "mask_cover": mask_cover(np.asarray(ref["R"]), np.asarray(ref["t"]), m, Kc,
                                                 model.dense_points, model.dense_normals)})
    results = dedup_detections(results, rc.get("dedup_mm", 0.0))
    rejected = []
    if rc.get("reject_unverified"):
        results, rejected = reject_unverified(results, crops, rc, offsets)
    flag_confident(results, rc)
    return results, rejected, len(crops)

In [ ]:
def pose_rows(dets, frame):
    """Per output detection: depth check (support / violation / occluded) and the confidence score (summary §10)."""
    acc = [d["icp_rmse"] for d in dets if d["icp_refined"]]
    med_rmse = float(np.median(acc)) if acc else np.nan                # frame's typical fit error
    rows = []
    for n, d in enumerate(dets):
        sc = surface_check(d["R"], d["t"], frame["depth"], frame["K"], model)
        rows.append({"det": n, "confident": d["confident"], "fit error (mm)": d["icp_rmse"], "fitness": d["icp_fitness"],
                     "support": sc["support"], "violation": sc["violation"], "occluded": sc["occluded"],
                     "conf score": (float(sc["support"] * d["icp_fitness"] * min(1.0, med_rmse / max(d["icp_rmse"], 1e-6)))
                                    if d["confident"] else 0.0)})
    return pd.DataFrame(rows).set_index("det")

def silhouette(det, frame):
    # projected CAD silhouette (all CAD points, closed and filled) in the image
    p = (model.dense_points @ np.asarray(det["R"]).T + np.asarray(det["t"])) @ np.asarray(frame["K"]).T
    u, v = np.round(p[:, 0] / p[:, 2]).astype(int), np.round(p[:, 1] / p[:, 2]).astype(int)
    h, w = frame["depth"].shape
    ok = (u >= 0) & (u < w) & (v >= 0) & (v < h)
    m = np.zeros((h, w), bool)
    m[v[ok], u[ok]] = True
    return binary_fill_holes(binary_closing(m, iterations=3))

def top_picks(df, dets, frame, k=TOP_K):
    # graspable parts (conditions 1-3) in confidence order, skipping any that overlaps a part already selected
    if not df.confident.any():
        return df.iloc[:0].assign(rank=[]), df.assign(**{"grasp ok": False})
    g = pd.DataFrame({n: grasp_check(dets[n], frame) for n in df.index[df.confident]}).T
    num = [c for c in g.columns if c not in ("approach side", "approach (cam)")]
    g[num] = g[num].astype(float)
    full = df.join(g)
    full["grasp ok"] = full.apply(lambda r: bool(r["confident"]) and why_not(r) == "", axis=1)
    sil, chosen = {}, []
    for n in full[full["grasp ok"]].sort_values("conf score", ascending=False).index:
        sil[n] = silhouette(dets[n], frame)
        a = max(sil[n].sum(), 1)
        if any((sil[n] & sil[m]).sum() / a > MAX_OVERLAP_SELECTED for m in chosen):
            continue
        chosen.append(n)
        if len(chosen) == k:
            break
    best = full.loc[chosen].copy()
    best.insert(0, "rank", range(1, len(best) + 1))
    return best, full


## 3. 실행: 촬영 → pose → 10개 (이 셀만 다시 실행하면 새로 찍습니다)

이 셀에 **hand-eye 행렬(`T_CAM_BASE`), 툴 방향 규칙, TCP 계산이 모두 들어 있어서** 이 셀만 다시 실행해도 전부 적용됩니다(모델은 위 셀에서 한 번 불러와 있어야 합니다).

- **좌표계:** robot base, 단위 m. `T_CAM_BASE`(카메라 → base, m)를 이 셀에서 정의하고 `camera/T_base_cam.json`보다 우선합니다. `p_base = T_CAM_BASE @ p_cam`.
- **TCP 목표 = 손끝 위치:** 로봇 설정 `tcp_offset_m: [0, 0, 0.106]`이 플랜지 → 손끝 길이이므로, TCP 목표는 손끝이 갈 점입니다. 위쪽 끝면의 가운데 구멍 입구에서 툴 z 방향으로 `INSERT_DEPTH`(8 mm) 들어간 점입니다(부품이 평평하면 원점보다 약 2.5 mm 위). `part x/y/z`가 아니라 `tcp x/y/z`를 로봇에 넣으세요.
- **툴 z축 = 부품 축(CAD z)을 따라 아래 방향:** 부품 z축이 위를 향하면 툴 z = −부품 z, 아래를 향하면 툴 z = +부품 z. 기울어진 부품도 같은 규칙이라 툴이 부품 축을 따라 들어갑니다. 구멍/그리퍼 주변 검사(`grasp_check`)도 같은 기준(base 위쪽 끝)으로 면을 고릅니다. **기울기 한도 `MAX_TILT` = 60°**(이 셀): 더 기운 부품은 `tilted`로 빠집니다. 90으로 두면 한도가 없습니다. **툴이 축 둘레로 도는 정도(`GRIP_YAW`, 이 셀):** 기본 `"down"`은 툴이 똑바로 아래를 보는 자세 rpy (180, 0, 0)에서 툴 z를 부품 축에 맞추는 **최소 회전**만 합니다. 평평한 부품은 항상 (180, 0, 0), 기울어진 부품은 기울기만큼만 기울고 축 둘레로 돌지 않습니다 (손가락이 둥근 가운데 구멍에 들어가므로 부품 x축 방향은 상관없음). `"part"`로 바꾸면 예전처럼 툴 x축을 부품 x축에 맞춥니다.
- **rpy 읽는 법:** 툴이 똑바로 아래를 보는 자세가 rpy (180, 0, 0)입니다. 그래서 기울어진 부품의 roll은 기울기 각도가 아니라 **180 ± 기울기** 근처로 나옵니다(예: 36.7° 기울기 → roll −143.9). 실제 기울기와 방향은 각 pick 줄의 `tool tilted … deg, tip moving toward (x, y)`(손끝이 내려가면서 움직이는 base 방향)로 보세요.
- **ee_6d 입력:** 각 pick 아래의 `ee_6d rpy :` 또는 `ee_6d quat:` 줄을 **그대로** `python3 ~/Projects/lite6_control/git/lite6_ws/src/my_lite6_control/my_lite6_control/ee_6d.py`에 붙여 넣으세요. 표와 `picks.json`의 rpy는 모두 `ee_6d.py`와 같은 intrinsic XYZ(R = Rx·Ry·Rz)이고, 쿼터니언은 (qx, qy, qz, qw) 순서입니다. `ros2 run my_lite6_control ee_6d`(예전 빌드, extrinsic xyz)는 쓰지 마세요.
- **part x/y/z, part roll/pitch/yaw:** 부품 원점의 자세(같은 intrinsic XYZ)입니다. 로봇에 넣는 값이 아니라 참고용입니다.
- **박스 테두리:** 파란 박스를 색으로 찾아, 부품 중심보다 높은 파란 점(테두리와 안쪽 벽면)까지 수평 거리가 `BIN_WALL_MARGIN` 10 cm 안이면 pick에서 뺍니다(`near bin wall`, 표의 `wall dist (m)`). 찾은 박스는 결과 그림에 청록색으로 표시합니다. 박스를 못 찾으면 이 규칙은 꺼지고 그렇다고 출력합니다. 화면 밖에 있는 벽(지금은 왼쪽)은 거리에 들어가지 않습니다.
- **도달 거리:** TCP 목표가 base 원점에서 `SAFE_REACH` 0.44 m보다 먼 부품은 pick에서 뺍니다(`out of reach`). 그 자리는 다음 부품이 채웁니다.
- **robot safety:** 로봇 쪽 안전 범위(x 0.05~0.45, y −0.45~0.45, z 0.01~0.60) 밖이면 이유를 적습니다. 그런 pick은 로봇이 거부하고 자동 집기에서 건너뜁니다.
- **그림:** 노란 번호 = pick 순서, 노란 점 = 원점. 박스는 고른 10개만 그립니다.


In [ ]:
from scipy.ndimage import binary_opening, label

# --- robot: hand-eye + TCP target. Defined here so that re-running only this cell applies them ---
# hand-eye calibration, camera -> robot base: p_base = T_CAM_BASE @ p_cam (metres)
# ~/Projects/xarm_lite6/cam2robot_pos_calibration/result/T_base_cam.yaml (touch/board_model, 2026-10-27 16:45,
# fit RMS 1.1 mm); copy the new matrix here after every recalibration
T_CAM_BASE = np.array([
    [-0.00169203, -0.99945118,  0.03308287,  0.25562417],
    [-0.97936149,  0.00834258,  0.20194425, -0.25696641],
    [-0.20210942, -0.03205840, -0.97883811,  0.71477754],
    [ 0.0,         0.0,         0.0,         1.0],
])
T_BASE_CAM = T_CAM_BASE.copy()
T_BASE_CAM[:3, 3] *= 1000.0                                          # the pipeline works in mm
FRAME, T_OUT = "base", T_BASE_CAM
UP_CAM = T_OUT[2, :3]                                                # base +Z in the camera frame (tilt reference)
R_TOOL_DOWN = Rotation.from_euler("xyz", [180, 0, 0], degrees=True).as_matrix()   # tool z straight down
# robot safety box for /target_pose (lite6_control config/default.yaml `safety`): TCP outside it is rejected there
SAFE_X, SAFE_Y, SAFE_Z, SAFE_REACH = (0.05, 0.45), (-0.45, 0.45), (0.01, 0.60), 0.44
# tilted parts are picked too: the tool goes in along the part axis, pointing down, whichever end is up
MAX_TILT = 60.0         # deg, part axis vs base +Z; parts tilted more are skipped ("tilted"); 90 = no limit
# turn of the tool about its own axis: "down" = the tool-down pose R_TOOL_DOWN turned as little as possible to put the
# tool z on the part axis (no spin: flat parts always get rpy 180 0 0; the fingers go into the round centre hole, so
# the part's own x does not matter) | "part" = tool x follows the part x (of +-x the one closer to tool-down)
GRIP_YAW = "down"
# blue bin: parts closer than BIN_WALL_MARGIN (horizontal, robot base) to the bin wall are skipped ("near bin wall"),
# the gripper would hit it. Wall = blue pixels (PIL HSV 0..255) higher than the part centre - BIN_WALL_BELOW: the rim
# and the inner wall faces that stick up around the part (blue at floor level outside the bin does not count).
BIN_WALL_MARGIN = 0.10  # m
BIN_HUE, BIN_MIN_SAT, BIN_MIN_VAL = (130, 185), 120, 35   # bluish glints on the parts stay below saturation ~110
BIN_WALL_BELOW = 0.01   # m
BIN_MIN_PX = 3000       # blue regions smaller than this are dropped; none left = no bin, the wall rule is off


def bin_wall(frame):
    """Blue bin pixels: image mask and their points in the robot base (m); None when no bin is seen."""
    hsv = np.asarray(Image.fromarray(frame["rgb"]).convert("HSV")).astype(int)
    blue = ((hsv[..., 0] >= BIN_HUE[0]) & (hsv[..., 0] <= BIN_HUE[1]) & (hsv[..., 1] >= BIN_MIN_SAT)
            & (hsv[..., 2] >= BIN_MIN_VAL) & frame["valid"])
    lab, _ = label(binary_opening(blue))                             # keep the large blue regions (rim, walls)
    size = np.bincount(lab.ravel())
    size[0] = 0
    blue = np.isin(lab, np.nonzero(size >= BIN_MIN_PX)[0])
    if not blue.any():
        return None
    p = (frame["xyz"][blue].astype(float) @ T_OUT[:3, :3].T + T_OUT[:3, 3]) / 1000.0   # camera mm -> base m
    return {"mask": blue, "points": p, "top_z": float(np.percentile(p[:, 2], 95))}


def wall_dist(det, frame):
    """Horizontal distance (m, robot base) from the part centre to the nearest bin wall point above it
    (inf = no wall that high, nan = no bin in the frame)."""
    wall = frame.get("wall")
    if wall is None:
        return float("nan")
    c = (T_OUT[:3, :3] @ np.asarray(det["t"], float) + T_OUT[:3, 3]) / 1000.0
    p = wall["points"]
    p = p[p[:, 2] > c[2] - BIN_WALL_BELOW]
    return float(np.sqrt(((p[:, :2] - c[:2]) ** 2).sum(1).min())) if len(p) else float("inf")


def grasp_check(det, frame):
    # entrance of the centre hole on the upper side (base frame), part axis, and what the measured cloud has around it.
    # The upper side is the end the part axis points to in the base frame, the same rule as tcp_pose, so the approach
    # (-up) is the tool z of the TCP target for upright, upside-down and tilted parts alike.
    R, t = np.asarray(det["R"], float), np.asarray(det["t"], float)
    hub = R[:, 2]
    side = 1.0 if hub @ UP_CAM >= 0 else -1.0                        # +1: hub end points up (base +Z)
    z_top = model.xyz_min[2] + model.xyz_size[2] if side > 0 else model.xyz_min[2]
    entry = R @ np.array([0.0, 0.0, z_top]) + t
    up = side * hub                                                  # out of the part, upwards
    tilt = float(np.degrees(np.arccos(np.clip(up @ UP_CAM, -1, 1))))
    K, xyz, valid = frame["K"], frame["xyz"], frame["valid"]
    c = K @ entry
    u0, v0 = c[0] / c[2], c[1] / c[2]
    half = int(1.5 * (R_BODY + 5) * K[0, 0] / entry[2]) + 2
    hh, ww = valid.shape
    ys = slice(max(int(v0) - half, 0), min(int(v0) + half, hh))
    xs = slice(max(int(u0) - half, 0), min(int(u0) + half, ww))
    d = xyz[ys, xs][valid[ys, xs]].astype(float) - entry
    h = d @ up                                                       # height above the entrance plane
    rho = np.linalg.norm(d - h[:, None] * up, axis=1)                # distance from the hole axis
    # blocked AREA share: blocking points / pixels the disc or ring covers in the image
    px_per_mm2 = (K[0, 0] / entry[2]) * (K[1, 1] / entry[2]) * max(abs(up[2]), 0.2)
    hole_px = np.pi * R_FINGER ** 2 * px_per_mm2
    ring_px = np.pi * (R_BODY ** 2 - R_FINGER ** 2) * px_per_mm2
    hole_blocked = min(float(((rho < R_FINGER) & (h > -INSERT_DEPTH)).sum()) / hole_px, 1.0)
    ring_blocked = min(float(((rho >= R_FINGER) & (rho < R_BODY) & (h > RING_MARGIN) & (h < APPROACH_LEN)).sum())
                       / ring_px, 1.0)
    return {"approach side": "hub" if side > 0 else "plate", "approach tilt (deg)": tilt,
            "hole blocked": hole_blocked, "ring blocked": ring_blocked, "approach (cam)": (-up).tolist(),
            "reach (m)": float(np.linalg.norm(tcp_pose(det)[1][:3, 3])),     # |TCP target| from the robot base
            "wall dist (m)": wall_dist(det, frame)}


def tcp_pose(det):
    """Part pose and robot TCP target in the robot base frame (metres).
    TCP = finger tip (the robot config's tcp_offset_m, 0.106 m, is the flange -> finger tip length): INSERT_DEPTH mm
    past the centre-hole entrance of the upper end, along the tool z. Tool z runs along the part axis (CAD z) and
    points DOWN in the base frame:
    part z up -> tool z = -part z, part z down -> tool z = +part z, and the same rule when the part is tilted.
    Rotation about the tool z: GRIP_YAW "down" = the shortest rotation taking the tool-down pose (rpy 180, 0, 0) onto
    that tool z, so the wrist only tilts and never spins with the part; "part" = tool x = part x (the part is 180 deg
    symmetric about its axis, so of +-x the one closer to tool-down)."""
    T_obj = np.eye(4)
    T_obj[:3, :3], T_obj[:3, 3] = np.asarray(det["R"]), np.asarray(det["t"])
    T_obj = T_OUT @ T_obj
    T_obj[:3, 3] /= 1000.0                                           # mm -> m
    part_z_up = bool(T_obj[2, 2] >= 0)
    z = -T_obj[:3, 2] if part_z_up else T_obj[:3, 2]
    if GRIP_YAW == "part":
        x = T_obj[:3, 0]
        R_tcp = min((np.column_stack([s * x, np.cross(z, s * x), z]) for s in (1.0, -1.0)),
                    key=lambda R: Rotation.from_matrix(R_TOOL_DOWN.T @ R).magnitude())
    else:
        z0 = R_TOOL_DOWN[:, 2]                                       # (0, 0, -1); z always points down, never opposite
        axis = np.cross(z0, z)
        angle = np.arctan2(np.linalg.norm(axis), z0 @ z)
        R_align = (Rotation.from_rotvec(axis / np.linalg.norm(axis) * angle).as_matrix()
                   if np.linalg.norm(axis) > 1e-9 else np.eye(3))
        R_tcp = R_align @ R_TOOL_DOWN
    z_top = model.xyz_min[2] + model.xyz_size[2] if part_z_up else model.xyz_min[2]   # upper end face (CAD z, mm)
    entry = T_obj[:3, 3] + T_obj[:3, 2] * z_top / 1000.0                               # hole entrance
    T_tcp = np.eye(4)
    T_tcp[:3, :3], T_tcp[:3, 3] = R_tcp, entry + z * INSERT_DEPTH / 1000.0
    return T_obj, T_tcp, part_z_up


def safety_issue(p):
    """Why the robot's safety box would reject TCP position p ("" = inside)."""
    for name, v, (lo, hi) in (("x", p[0], SAFE_X), ("y", p[1], SAFE_Y), ("z", p[2], SAFE_Z)):
        if not lo <= v <= hi:
            return f"{name} {v:.3f} outside [{lo}, {hi}]"
    if np.linalg.norm(p) > SAFE_REACH:
        return f"reach {np.linalg.norm(p):.3f} > {SAFE_REACH}"
    return ""


# pick conditions (used by top_picks); defined here so that re-running only this cell applies them
def why_not(r, detail=True):
    # first failed condition ("" = graspable); detail=False gives just the category
    if not r["confident"]:
        return "not confident"
    for bad, name, val in ((r["reach (m)"] > SAFE_REACH, "out of reach", f"{r['reach (m)']:.3f} m"),
                           (r["wall dist (m)"] < BIN_WALL_MARGIN, "near bin wall", f"{r['wall dist (m)']:.3f} m"),
                           (r["occluded"] > GRASP_MAX_OCCLUDED, "covered", f"occluded {r['occluded']:.2f}"),
                           (r["approach tilt (deg)"] > MAX_TILT, "tilted", f"{r['approach tilt (deg)']:.0f} deg"),
                           (r["hole blocked"] > HOLE_BLOCK_MAX, "hole blocked", f"{r['hole blocked']:.0%}"),
                           (r["ring blocked"] > RING_BLOCK_MAX, "gripper ring blocked", f"{r['ring blocked']:.0%}")):
        if bad:
            return f"{name} {val}" if detail else name
    return ""

def run_cycle(save=True):
    t0 = time.perf_counter()
    f = capture_frame()
    f["wall"] = bin_wall(f)
    t1 = time.perf_counter()
    dets, rejected, n_masks = estimate_poses(f)
    t2 = time.perf_counter()
    stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    print(f"{f['source']} | capture {t1 - t0:.1f} s, pose {t2 - t1:.1f} s | "
          + (f"blue bin found (rim z {f['wall']['top_z']:.3f} m), skipping parts within {BIN_WALL_MARGIN} m of its wall"
             if f["wall"] else "NO blue bin found: the wall rule is off"))
    if not dets:
        print("no verified side plate in this frame")
        return pd.DataFrame()
    df = pose_rows(dets, f)
    best, full = top_picks(df, dets, f)
    print(f"{n_masks} detector masks -> {len(dets)} verified ({len(rejected)} rejected) -> {int(df.confident.sum())} confident "
          f"-> {int(full['grasp ok'].sum())} graspable -> {len(best)} selected | frame: {FRAME}")
    reasons = full[full.confident.astype(bool)].apply(lambda r: why_not(r, detail=False), axis=1, result_type="reduce")
    if (reasons != "").any():
        print("confident but not graspable:", ", ".join(f"{k} {v}" for k, v in reasons[reasons != ""].value_counts().items()))
    if best.empty:
        print("no graspable side plate in this frame")
        return pd.DataFrame()

    # Poses for ee_6d.py (src): robot base, metres; "x y z roll pitch yaw" is read as INTRINSIC XYZ (R = Rx Ry Rz),
    # "x y z qx qy qz qw" as a quaternion. Angles of another convention would tilt the tool the wrong way.
    rows, out = [], []
    for n, r in best.iterrows():
        T_obj, T_tcp, part_z_up = tcp_pose(dets[n])
        rot = Rotation.from_matrix(T_tcp[:3, :3])
        rpy = rot.as_euler("XYZ", degrees=True)
        part_rpy = Rotation.from_matrix(T_obj[:3, :3]).as_euler("XYZ", degrees=True)
        qx, qy, qz, qw = rot.as_quat()
        safety = safety_issue(T_tcp[:3, 3])
        rows.append({"rank": r["rank"], "det": n, "conf score": r["conf score"],
                     "tcp x (m)": T_tcp[0, 3], "tcp y (m)": T_tcp[1, 3], "tcp z (m)": T_tcp[2, 3],
                     "tcp roll (deg)": rpy[0], "tcp pitch (deg)": rpy[1], "tcp yaw (deg)": rpy[2],
                     "qx": qx, "qy": qy, "qz": qz, "qw": qw,
                     "part x (m)": T_obj[0, 3], "part y (m)": T_obj[1, 3], "part z (m)": T_obj[2, 3],
                     "part roll (deg)": part_rpy[0], "part pitch (deg)": part_rpy[1], "part yaw (deg)": part_rpy[2],
                     "part axis": "up" if part_z_up else "down", "tool z (base)": np.round(T_tcp[:3, 2], 4).tolist(),
                     "tool tilt (deg)": float(np.degrees(np.arccos(np.clip(-T_tcp[2, 2], -1, 1)))),
                     "approach tilt (deg)": r["approach tilt (deg)"], "hole blocked": r["hole blocked"],
                     "ring blocked": r["ring blocked"], "wall dist (m)": r["wall dist (m)"], "occluded": r["occluded"],
                     "fit error (mm)": r["fit error (mm)"], "robot safety": safety or "ok"})
        out.append({"rank": int(r["rank"]), "conf_score": float(r["conf score"]),
                    "tcp_position_m": np.round(T_tcp[:3, 3], 6).tolist(),
                    "tcp_rpy_intrinsic_XYZ_deg": np.round(rpy, 3).tolist(),
                    "tcp_quat_xyzw": np.round([qx, qy, qz, qw], 6).tolist(), "T_base_tcp": np.round(T_tcp, 6).tolist(),
                    "part_position_m": np.round(T_obj[:3, 3], 6).tolist(),
                    "part_rpy_intrinsic_XYZ_deg": np.round(part_rpy, 3).tolist(), "T_base_part": np.round(T_obj, 6).tolist(),
                    "part_axis": "up" if part_z_up else "down", "robot_safety": safety or "ok"})
    picks = pd.DataFrame(rows).set_index("rank")

    print("\nTCP target in the robot base. Paste ONE of the two lines into ee_6d.py:")
    for k, r in picks.iterrows():
        xyz = f"{r['tcp x (m)']:.4f} {r['tcp y (m)']:.4f} {r['tcp z (m)']:.4f}"
        tz = r["tool z (base)"]
        print(f"#{k:<2d} score {r['conf score']:.3f} | part axis {r['part axis']:4s} -> tool z down "
              f"| tool tilted {r['tool tilt (deg)']:.1f} deg from straight down, tip moving toward "
              f"({tz[0]:+.2f} x, {tz[1]:+.2f} y) of the base | safety: {r['robot safety']}")
        print(f"    ee_6d rpy : {xyz} {r['tcp roll (deg)']:.2f} {r['tcp pitch (deg)']:.2f} {r['tcp yaw (deg)']:.2f}")
        print(f"    ee_6d quat: {xyz} {r['qx']:.6f} {r['qy']:.6f} {r['qz']:.6f} {r['qw']:.6f}")
    display(picks.round(4))

    # rank labels at each origin
    rgb_vis = f["rgb"].copy()
    if f["wall"]:                                                    # detected blue bin in cyan
        rgb_vis[f["wall"]["mask"]] = (0.5 * rgb_vis[f["wall"]["mask"]] + (0, 127, 127)).astype(np.uint8)
    img = render_2d(rgb_vis, f["K"], [d if n in set(best.index) else None for n, d in enumerate(dets)], model, None,
                    labels="none", title=f"{len(best)} picks (number = pick order)")
    dr = ImageDraw.Draw(img)
    font = ImageFont.load_default(size=28)
    for n, r in best.iterrows():
        c = np.asarray(f["K"]) @ np.asarray(dets[n]["t"])
        u, v = c[0] / c[2], c[1] / c[2]
        dr.ellipse((u - 5, v - 5, u + 5, v + 5), fill=(255, 230, 0))
        dr.text((u + 8, v - 30), str(int(r["rank"])), fill=(255, 230, 0), font=font, stroke_width=3, stroke_fill=(0, 0, 0))
    display(img.resize((1000, int(1000 * img.height / img.width))))

    if save:
        d = OUT_DIR / stamp
        d.mkdir(parents=True, exist_ok=True)
        json.dump({"source": f["source"], "frame": "base", "unit": "m", "rpy": "intrinsic XYZ (ee_6d.py)",
                   "T_cam_base": T_CAM_BASE.tolist(),
                   "tcp": f"finger tip {INSERT_DEPTH} mm inside the upper hole entrance, tool z along the part axis pointing down", "picks": out},
                  open(d / "picks.json", "w"), indent=1)
        p = picks.copy()
        p["tool z (base)"] = p["tool z (base)"].map(lambda a: " ".join(f"{x:.4f}" for x in a))
        p.to_csv(d / "picks.csv", float_format="%.6f")
        json.dump(dets, open(d / "detections.json", "w"))
        img.save(d / "overlay.png")
        Image.fromarray(f["rgb"]).save(d / "rgb.png")
        np.save(d / "depth.npy", f["depth"].astype(np.float32))
        print(f"saved -> {d}")
    return picks

picks = run_cycle()


### 자동 집기 (위 셀의 pick 1~10번)

위 셀이 찍고 계산한 `picks`의 1~10번을 차례대로 `ee_6d.py`의 publisher로 `/target_pose`에 보냅니다. 로봇이 한 번 집고 **home으로 돌아오면** 다음 번호를 보냅니다.

- 먼저 터미널 1(MoveIt + 로봇)과 터미널 2(`sequence_real_action`)를 띄워 두세요. 터미널 3·4는 필요 없습니다.
- `robot safety`가 ok가 아닌 번호는 보내지 않습니다. 로봇이 움직이지 않은 번호(안전 범위·IK 거부)나 도중에 멈춘 번호는 실패로 표시하고 다음 번호로 넘어갑니다.
- 셀을 멈추면(■) 다음 번호를 보내지 않습니다. 이미 시작한 동작은 `sequence_real_action`이 끝까지 합니다. 바로 멈추려면 터미널 2를 Ctrl+C 하세요.
- 같은 촬영 결과로 10개를 집으므로, 앞에서 집을 때 다른 부품이 밀리면 뒤 번호는 빗나갈 수 있습니다.


In [ ]:
# --- auto pick: ranks AUTO_PICK_RANKS of `picks` (cell above), one after another through ee_6d.py -> /target_pose
# -> sequence_real_action; the next one is sent when the arm is back at its home pose (auto_pick_ee6d.py)
AUTO_PICK_RANKS = list(range(1, 11))   # 1..10 (TOP_K picks at most)
EE_6D = Path.home() / "Projects/lite6_control/git/lite6_ws/src/my_lite6_control/my_lite6_control/ee_6d.py"
ROS_SETUP = ("source /opt/ros/jazzy/setup.bash && source ~/Projects/lite6_control/git/xarm_ros2/install/setup.bash "
             "&& source ~/Projects/lite6_control/git/lite6_ws/install/setup.bash")


def auto_pick(picks, ranks=AUTO_PICK_RANKS):
    if picks is None or picks.empty:
        print("no picks: run the cell above first")
        return
    targets = []
    for k in ranks:
        if k not in picks.index:
            print(f"#{k}: not in this run's picks ({len(picks)} picks), skipped")
            continue
        r = picks.loc[k]
        if r["robot safety"] != "ok":
            print(f"#{k}: robot safety {r['robot safety']}, skipped")
            continue
        targets.append({"rank": int(k), "xyz": [float(r["tcp x (m)"]), float(r["tcp y (m)"]), float(r["tcp z (m)"])],
                        "quat_xyzw": [float(r[c]) for c in ("qx", "qy", "qz", "qw")]})
    if not targets:
        print("nothing to pick")
        return
    path = OUT_DIR / "auto_pick_targets.json"
    path.parent.mkdir(parents=True, exist_ok=True)
    json.dump({"frame": "base", "unit": "m", "targets": targets}, open(path, "w"), indent=1)
    print(f"picking {[t['rank'] for t in targets]} ({path})")

    # ROS python, not this kernel's .venv
    env = {k: v for k, v in os.environ.items() if k not in ("VIRTUAL_ENV", "PYTHONHOME", "PYTHONPATH")}
    cmd = f"{ROS_SETUP} && /usr/bin/python3 -u {Path('binpick/auto_pick_ee6d.py').resolve()} {path.resolve()} --ee6d {EE_6D}"
    proc = subprocess.Popen(["bash", "-c", cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=env)
    try:
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
    except KeyboardInterrupt:
        proc.send_signal(2)  # SIGINT: stop sending targets, print the summary
        for line in proc.stdout:
            print(line, end="")
        proc.wait()


auto_pick(picks)
